# Customer Churn Prediction — AUROC Model Comparison

**Graduate machine learning portfolio | Python · scikit-learn · classification · model evaluation**

This notebook compares five supervised models for predicting churn from customer demographics, account tenure, subscribed services, and billing attributes. An **80/20 stratified split** estimates generalization before generating probabilities for the unlabeled Kaggle competition set.

**Best validation AUROC: 0.8381 — Logistic Regression** (not a verified Kaggle leaderboard score). There are 5,634 labeled customers (4,507 training / 1,127 validation) and 1,409 unlabeled test customers.

The original competition files are not redistributed. See the accompanying README for instructions and limitations.


In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# Runs on Kaggle or locally with data/train.csv and data/test.csv.
KAGGLE_DATA_DIR = "/kaggle/input/competitions/customer-churn-prediction-fall-2026-idc-6210"
DATA_DIR = os.environ.get(
    "CHURN_DATA_DIR",
    KAGGLE_DATA_DIR if os.path.isdir(KAGGLE_DATA_DIR) else "data"
)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/customer-churn-prediction-fall-2026-idc-6210/sample_submission.csv
/kaggle/input/competitions/customer-churn-prediction-fall-2026-idc-6210/train.csv
/kaggle/input/competitions/customer-churn-prediction-fall-2026-idc-6210/test.csv


## Part 1: Train model

In [19]:
# Load the training data
df_traindata = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_traindata.head()

,id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,label
0,1815,Male,0,Yes,Yes,12,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.70,258.35,0
1,5947,Female,0,No,No,42,Yes,No,DSL,Yes,...,Yes,Yes,No,Yes,One year,No,Credit card (automatic),73.90,3160.55,1
2,3882,Male,0,Yes,No,71,Yes,Yes,DSL,Yes,...,No,Yes,No,No,Two year,No,Bank transfer (automatic),65.15,4681.75,0
3,2390,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,No,Electronic check,85.45,6300.85,0
4,3677,Male,0,No,No,30,Yes,No,DSL,Yes,...,No,Yes,Yes,No,One year,No,Electronic check,70.40,2044.75,0


### Splitting data to features and labels

In [20]:
# Split the data into features (X) and target variable (y)
# Hint: training model doesn't require id or target variable

X = df_traindata.drop(columns=['id', 'label'])
y = df_traindata['label']

print("Feature matrix shape:", X.shape)
print("Target variable shape:", y.shape)

X.head()
y.head()

Feature matrix shape: (5634, 19)
Target variable shape: (5634,)


0    0
1    1
2    0
3    0
4    0
Name: label, dtype: int64

We split customer data into the feature matrix **X** (demographics, services, account tenure, and billing attributes) and the known churn outcomes **y**. We exclude `label` from the predictors to prevent target leakage, and omit `id` because an arbitrary customer identifier should not be used as a predictive feature.


## Splitting dataset to training and validation sets

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Training labels:", y_train.shape)
print("Validation labels:", y_val.shape)

print("\nOverall churn rate:", round(y.mean(), 3))
print("Training churn rate:", round(y_train.mean(), 3))
print("Validation churn rate:", round(y_val.mean(), 3))

Training features: (4507, 19)
Validation features: (1127, 19)
Training labels: (4507,)
Validation labels: (1127,)

Overall churn rate: 0.27
Training churn rate: 0.27
Validation churn rate: 0.27


We reserve **80%** of labeled customer records (4,507) for model training and **20%** (1,127) for validation. The validation customers are not included in fitting the models.

The `stratify=y` argument preserves the approximately **27% churn rate** in both sets. `random_state=42` makes the random split reproducible.


## Feature engineering for training and validation sets

In [22]:
from sklearn.preprocessing import StandardScaler

X_train = X_train.copy()
X_val = X_val.copy()

X_train['total_charges'] = pd.to_numeric(
    X_train['total_charges'], errors='coerce'
)
X_val['total_charges'] = pd.to_numeric(
    X_val['total_charges'], errors='coerce'
)

median_charges = X_train['total_charges'].median()
X_train['total_charges'] = X_train['total_charges'].fillna(median_charges)
X_val['total_charges'] = X_val['total_charges'].fillna(median_charges)

X_train_encoded = pd.get_dummies(X_train, dtype=int)
X_val_encoded = pd.get_dummies(X_val, dtype=int)

X_val_encoded = X_val_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_val_scaled = scaler.transform(X_val_encoded)

The `total_charges` field is stored as text, so we convert it to numeric and fill missing or non-convertible values with the **training-set median**. That same value is reused for the validation set rather than deriving statistics from validation records.

We one-hot encode categorical features with `pd.get_dummies()`, then use `reindex()` so the training and validation sets have the same columns. `StandardScaler` learns feature means and standard deviations from the training set and applies those parameters to both sets. Tree-based models use the aligned encoded (unscaled) features.


## Building models

In [23]:
from sklearn.linear_model import Perceptron

pct_model = Perceptron(
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

In [24]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=2000,
    C=1.0,
    random_state=42
)

In [25]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=7,
    weights='distance'
)

In [26]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=10,
    random_state=42
)

In [27]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

We build five classifiers to compare different modeling approaches:

- **Perceptron:** learns a linear decision boundary.
- **Logistic Regression:** estimates churn probabilities using a regularized linear classifier.
- **K-Nearest Neighbors (KNN):** uses nearby customers, with closer neighbors weighted more heavily.
- **Decision Tree:** learns nonlinear decision rules, with depth and leaf constraints to moderate overfitting.
- **Random Forest:** averages 300 randomized decision trees to capture interactions and reduce single-tree variance.

These are **initial hyperparameter choices**, not the result of an exhaustive optimization search.


## Training models with training set features and labels

In [28]:
# Fit pct_model - Example: pct_model.fit()
pct_model.fit(X_train_scaled, y_train)

Perceptron(random_state=42)

In [29]:
# Fit log_model
log_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=2000, random_state=42)

In [30]:
# Fit knn_model
knn_model.fit(X_train_scaled, y_train)

KNeighborsClassifier(n_neighbors=7, weights='distance')

In [31]:
# Fit dt_model
dt_model.fit(X_train_encoded, y_train)

DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42)

In [32]:
# Fit rf_model
rf_model.fit(X_train_encoded, y_train)

RandomForestClassifier(min_samples_leaf=2, n_estimators=300, n_jobs=-1,
                       random_state=42)

## Predict **label probability** with validation set features

In [34]:
#Remember, AUROC is calculated by sliding a threshold over the dataset probabilities, observing how well each choice of threshold separates the two classes!
#If you use **predicted labels** instead of **label probabilities**, your model will have much less performance.

# Perceptron does not support predict_proba; use continuous decision scores for AUROC.
pct_pred = pct_model.decision_function(X_val_scaled)

# Logistic Regression predictions
log_pred = log_model.predict_proba(X_val_scaled)[:, 1]

# KNN predictions
knn_pred = knn_model.predict_proba(X_val_scaled)[:, 1]

# Decision Tree predictions
dt_pred = dt_model.predict_proba(X_val_encoded)[:, 1]

# Random Forest predictions
rf_pred = rf_model.predict_proba(X_val_encoded)[:, 1]

We apply our trained models to validation features. Logistic Regression, KNN, Decision Tree, and Random Forest use `predict_proba()[:, 1]` to obtain probabilities for the positive class (churn).

The Perceptron cannot output probabilities, so `decision_function()` provides continuous ranking scores instead. AUROC measures discrimination across thresholds and does not require calibrated probabilities.


## Evaluating model (validation set ground truth labels vs validation set probabilities)

In [35]:
from sklearn.metrics import roc_auc_score

# Perceptron AUROC
pct_auroc = roc_auc_score(y_val, pct_pred)

# Logistic Regression AUROC
log_auroc = roc_auc_score(y_val, log_pred)

# KNN AUROC
knn_auroc = roc_auc_score(y_val, knn_pred)

# Decision Tree AUROC
dt_auroc = roc_auc_score(y_val, dt_pred)

# Random Forest AUROC
rf_auroc = roc_auc_score(y_val, rf_pred)

print(f"Perceptron AUROC: {pct_auroc:.4f}")
print(f"Logistic Regression AUROC: {log_auroc:.4f}")
print(f"KNN AUROC: {knn_auroc:.4f}")
print(f"Decision Tree AUROC: {dt_auroc:.4f}")
print(f"Random Forest AUROC: {rf_auroc:.4f}")

Perceptron AUROC: 0.7527
Logistic Regression AUROC: 0.8381
KNN AUROC: 0.7598
Decision Tree AUROC: 0.8267
Random Forest AUROC: 0.8257


### Model evaluation: holdout AUROC

We calculate `roc_auc_score()` against the true validation labels. A score of 0.5 represents random ranking, and 1.0 represents perfect discrimination.

| Classifier | Validation AUROC |
| --- | ---: |
| **Logistic Regression** | **0.8381** |
| Decision Tree | 0.8267 |
| Random Forest | 0.8257 |
| K-Nearest Neighbors | 0.7598 |
| Perceptron | 0.7527 |

**Why Logistic Regression?** It achieved the highest observed validation AUROC among our five candidates. Despite being simpler than the tree-based models, it ranked churn risk more effectively on this split, so we select it for the unlabeled competition test set.

These are scores from **one held-out validation partition**, not verified Kaggle leaderboard results. No extensive hyperparameter search or nested validation was performed.


## Part 2: Test data

In [38]:
# Load the test data
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

df_test.head()

test = df_test.drop(columns=['id'])

We load the unlabeled competition test dataset and exclude customer `id` from the predictive features. We retain the original `df_test` table so we can attach each prediction to its corresponding ID when preparing the submission.


## Loading and preprocessing the test data
## Feature engineering for test set

In [43]:
test['total_charges'] = pd.to_numeric(
    test['total_charges'], errors='coerce'
).fillna(median_charges)

test = pd.get_dummies(test, dtype=int)
test = test.reindex(columns=X_train_encoded.columns, fill_value=0)

test_scaled = scaler.transform(test)

print("Test data before scaling:", test.shape)
print("Test data after scaling:", test_scaled.shape)
print("Missing values:", test.isna().sum().sum())

display(pd.DataFrame(
    test_scaled,
    columns=X_train_encoded.columns
).head())

Test data before scaling: (1409, 45)
Test data after scaling: (1409, 45)
Missing values: 0


,senior_citizen,tenure,monthly_charges,total_charges,gender_Female,gender_Male,partner_No,partner_Yes,dependents_No,dependents_Yes,...,streaming_movies_Yes,contract_Month-to-month,contract_One year,contract_Two year,paperless_billing_No,paperless_billing_Yes,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
0,-0.441426,0.344536,0.495898,0.451568,1.006455,-1.006455,0.958057,-0.958057,0.644260,-0.644260,...,1.238899,-1.099688,1.958529,-0.570945,-0.836344,0.836344,1.873903,-0.52609,-0.696074,-0.554539
1,2.265386,1.361474,1.246005,1.838973,1.006455,-1.006455,0.958057,-0.958057,0.644260,-0.644260,...,1.238899,-1.099688,-0.510587,1.751482,-0.836344,0.836344,1.873903,-0.52609,-0.696074,-0.554539
2,-0.441426,-0.835112,-0.663359,-0.779665,1.006455,-1.006455,0.958057,-0.958057,0.644260,-0.644260,...,-0.807168,0.909349,-0.510587,-0.570945,-0.836344,0.836344,1.873903,-0.52609,-0.696074,-0.554539
3,-0.441426,-1.119855,-0.477080,-0.900481,1.006455,-1.006455,0.958057,-0.958057,0.644260,-0.644260,...,-0.807168,0.909349,-0.510587,-0.570945,-0.836344,0.836344,-0.533646,-0.52609,-0.696074,1.803299
4,-0.441426,-0.916467,0.031862,-0.719917,1.006455,-1.006455,-1.043780,1.043780,-1.552169,1.552169,...,1.238899,-1.099688,1.958529,-0.570945,-0.836344,0.836344,-0.533646,-0.52609,-0.696074,1.803299


The test data contain text categories and `total_charges` stored as text. We convert charges into numbers, reuse the **training median** for missing entries, and one-hot encode categorical values. `reindex()` aligns the resulting features with the 45 encoded training columns.

We apply `scaler.transform()` using training-derived parameters rather than refitting on test data. Our stored execution output shows **1,409 test customers, 45 features, and zero missing values** after preparation.


## Testing model on real-world data (test data) that has no label
For this you only need to use one model that you have chosen to create the submission predictions.
REMEMBER: you must predict **probability** of a label, not the label itself 

In [45]:
test_pred = log_model.predict_proba(test_scaled)[:, 1]

prediction_results = pd.DataFrame({
    'id': df_test['id'],
    'churn_probability': test_pred
})

display(prediction_results.head(10))

,id,churn_probability
0,3382,0.075611
1,6181,0.101438
2,4830,0.386010
3,3738,0.459069
4,4250,0.216657
5,6836,0.131467
6,3870,0.334825
7,3944,0.492635
8,3492,0.075407
9,6848,0.235023


Logistic Regression was selected because it delivered the strongest observed validation AUROC (**0.8381**). Using `predict_proba(test_scaled)[:, 1]`, we generate churn probabilities for all **1,409** unlabeled customers.

The preview matches predictions with the original customer IDs. The final code cell writes `submission.csv` with exactly the two expected columns, **`id`** and **`label`**, where `label` contains a probability rather than a hard class prediction. A Kaggle leaderboard score is not included because it has not been verified.


In [46]:
df_submission = pd.DataFrame({'id': df_test['id'], 'label': test_pred})
df_submission.to_csv('submission.csv', index=False)
df_submission.head()

,id,label
0,3382,0.075611
1,6181,0.101438
2,4830,0.386010
3,3738,0.459069
4,4250,0.216657


### Limitations and next steps

The reported **0.8381 AUROC** is based on one stratified holdout split, not an independent Kaggle test evaluation. The final competition probabilities were generated by a model trained on the **4,507-record training partition**, not retrained on all 5,634 labeled examples. Cross-validated tuning and refitting on the full labeled dataset are potential follow-up improvements; they were not part of the results reported here.
